In [55]:
import pandas as pd
import numpy as np
import duckdb

import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("Pandas version:", pd.__version__)
print("DuckDB version:", duckdb.__version__)

Pandas version: 3.1.0
DuckDB version: 2.0.0


In [56]:
current_path = Path.cwd()

PROJECT_ROOT = None

for path in [current_path, *current_path.parents]:
    if (path / "data" / "raw" / "events.csv").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не удалось найти data/raw/events.csv. "
        "Проверь расположение проекта."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "events.csv"

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Tables directory:", TABLES_DIR)

Project root: /Users/hkkkkjv/Study/memory/4 course/boot/ecommerce-product-analytics
Data path: /Users/hkkkkjv/Study/memory/4 course/boot/ecommerce-product-analytics/data/raw/events.csv
Tables directory: /Users/hkkkkjv/Study/memory/4 course/boot/ecommerce-product-analytics/reports/tables


In [57]:
events = pd.read_csv(DATA_PATH)

events["timestamp"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    utc=True,
    errors="coerce"
)

events = events.dropna(
    subset=["timestamp", "visitorid", "itemid", "event"]
).copy()

events["visitorid"] = events["visitorid"].astype("int64")
events["itemid"] = events["itemid"].astype("int64")

events = events.sort_values(
    ["visitorid", "timestamp"]
).reset_index(drop=True)

print("Dataset shape:", events.shape)

display(events.head())

Dataset shape: (2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,2015-09-11 20:49:49.439+00:00,0,view,285930,NaN
1,2015-09-11 20:52:39.591+00:00,0,view,357564,NaN
2,2015-09-11 20:55:17.175+00:00,0,view,67045,NaN
3,2015-08-13 17:46:06.444+00:00,1,view,72028,NaN
4,2015-08-07 17:51:44.567+00:00,2,view,325215,NaN


In [58]:
data_overview = pd.DataFrame({
    "Metric": [
        "Total events",
        "Unique users",
        "Unique items",
        "Unique event types",
        "Missing timestamps",
        "Missing visitor IDs",
        "Missing item IDs",
        "Missing transaction IDs"
    ],
    "Value": [
        len(events),
        events["visitorid"].nunique(),
        events["itemid"].nunique(),
        events["event"].nunique(),
        events["timestamp"].isna().sum(),
        events["visitorid"].isna().sum(),
        events["itemid"].isna().sum(),
        events["transactionid"].isna().sum()
    ]
})

display(data_overview)

,Metric,Value
0,Total events,2756101
1,Unique users,1407580
2,Unique items,235061
3,Unique event types,3
4,Missing timestamps,0
5,Missing visitor IDs,0
6,Missing item IDs,0
7,Missing transaction IDs,2733644


In [59]:
data_overview = pd.DataFrame({
    "Metric": [
        "Total events",
        "Unique users",
        "Unique items",
        "Unique event types",
        "Missing timestamps",
        "Missing visitor IDs",
        "Missing item IDs",
        "Missing transaction IDs"
    ],
    "Value": [
        len(events),
        events["visitorid"].nunique(),
        events["itemid"].nunique(),
        events["event"].nunique(),
        events["timestamp"].isna().sum(),
        events["visitorid"].isna().sum(),
        events["itemid"].isna().sum(),
        events["transactionid"].isna().sum()
    ]
})

display(data_overview)

,Metric,Value
0,Total events,2756101
1,Unique users,1407580
2,Unique items,235061
3,Unique event types,3
4,Missing timestamps,0
5,Missing visitor IDs,0
6,Missing item IDs,0
7,Missing transaction IDs,2733644


In [60]:
event_distribution = (
    events["event"]
    .value_counts()
    .rename_axis("event")
    .reset_index(name="event_count")
)

event_distribution["percentage"] = (
    event_distribution["event_count"]
    / event_distribution["event_count"].sum()
    * 100
)

display(event_distribution)

,event,event_count,percentage
0,view,2664312,96.669607
1,addtocart,69332,2.515583
2,transaction,22457,0.814810


In [61]:
con = duckdb.connect(database=":memory:")

con.execute("SET TimeZone = 'UTC'")

con.register("events_df", events)

con.execute("""
    CREATE OR REPLACE TABLE events AS
    SELECT *
    FROM events_df
""")

table_info = con.execute("""
    DESCRIBE events
""").df()

display(table_info)

,column_name,column_type,null,key,default,extra
0,timestamp,TIMESTAMP WITH TIME ZONE,YES,None,None,None
1,visitorid,BIGINT,YES,None,None,None
2,event,VARCHAR,YES,None,None,None
3,itemid,BIGINT,YES,None,None,None
4,transactionid,DOUBLE,YES,None,None,None


In [62]:
sample_sql = con.execute("""
    SELECT *
    FROM events
    LIMIT 10
""").df()

display(sample_sql)

/var/folders/gk/l4rj7d3x7wj7wlw6z7x421mr0000gn/T/ipykernel_9414/112503734.py:5: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


,timestamp,visitorid,event,itemid,transactionid
0,2015-09-11 20:49:49.439+00:00,0,view,285930,NaN
1,2015-09-11 20:52:39.591+00:00,0,view,357564,NaN
2,2015-09-11 20:55:17.175+00:00,0,view,67045,NaN
3,2015-08-13 17:46:06.444+00:00,1,view,72028,NaN
4,2015-08-07 17:51:44.567+00:00,2,view,325215,NaN
5,2015-08-07 17:53:33.790+00:00,2,view,325215,NaN
6,2015-08-07 17:56:52.664+00:00,2,view,259884,NaN
7,2015-08-07 18:01:08.920+00:00,2,view,216305,NaN
8,2015-08-07 18:08:25.669+00:00,2,view,342816,NaN
9,2015-08-07 18:17:24.375+00:00,2,view,342816,NaN


In [63]:
event_sql = con.execute("""
    SELECT
        event,
        COUNT(*) AS event_count,
        COUNT(DISTINCT visitorid) AS unique_users,
        COUNT(DISTINCT itemid) AS unique_items
    FROM events
    GROUP BY event
    ORDER BY event_count DESC
""").df()

display(event_sql)

,event,event_count,unique_users,unique_items
0,view,2664312,1404179,234838
1,addtocart,69332,37722,23903
2,transaction,22457,11719,12025


In [64]:
event_sql.to_csv(
    TABLES_DIR / "sql_event_distribution.csv",
    index=False
)

In [65]:
user_activity = con.execute("""
    SELECT
        visitorid,

        COUNT(*) AS total_events,

        COUNT(*) FILTER (
            WHERE event = 'view'
        ) AS views,

        COUNT(*) FILTER (
            WHERE event = 'addtocart'
        ) AS addtocarts,

        COUNT(*) FILTER (
            WHERE event = 'transaction'
        ) AS transactions,

        COUNT(DISTINCT itemid) FILTER (
            WHERE event = 'view'
        ) AS unique_items_viewed

    FROM events
    GROUP BY visitorid
""").df()

display(user_activity.head())

,visitorid,total_events,views,addtocarts,transactions,unique_items_viewed
0,322249,1,1,0,0,1
1,322285,1,1,0,0,1
2,322330,2,2,0,0,1
3,322349,1,1,0,0,1
4,322389,1,1,0,0,1


In [66]:
user_segments = con.execute("""
    WITH user_metrics AS (
        SELECT
            visitorid,

            COUNT(*) FILTER (
                WHERE event = 'view'
            ) AS views,

            COUNT(*) FILTER (
                WHERE event = 'addtocart'
            ) AS addtocarts,

            COUNT(*) FILTER (
                WHERE event = 'transaction'
            ) AS transactions

        FROM events
        GROUP BY visitorid
    )

    SELECT
        CASE
            WHEN transactions > 0 THEN 'Buyer'
            WHEN addtocarts > 0 THEN 'Cart user'
            WHEN views > 0 THEN 'Viewer'
            ELSE 'Other'
        END AS user_segment,

        COUNT(*) AS users

    FROM user_metrics
    GROUP BY user_segment
    ORDER BY users DESC
""").df()

user_segments["percentage"] = (
    user_segments["users"]
    / user_segments["users"].sum()
    * 100
)

display(user_segments)

,user_segment,users,percentage
0,Viewer,1368715,97.238878
1,Cart user,27146,1.928558
2,Buyer,11719,0.832564


In [67]:
item_performance = con.execute("""
    SELECT
        itemid,

        COUNT(*) FILTER (
            WHERE event = 'view'
        ) AS views,

        COUNT(*) FILTER (
            WHERE event = 'addtocart'
        ) AS addtocarts,

        COUNT(*) FILTER (
            WHERE event = 'transaction'
        ) AS transaction_events,

        COUNT(DISTINCT visitorid) FILTER (
            WHERE event = 'view'
        ) AS unique_viewers,

        COUNT(DISTINCT transactionid) FILTER (
            WHERE event = 'transaction'
        ) AS unique_orders

    FROM events
    GROUP BY itemid
""").df()

display(item_performance.head())

,itemid,views,addtocarts,transaction_events,unique_viewers,unique_orders
0,389548,12,0,0,12,0
1,27229,100,2,0,78,0
2,47024,7,1,1,4,1
3,456155,2,0,0,2,0
4,305656,262,10,4,214,4


In [68]:
top_viewed_items = (
    item_performance
    .sort_values("views", ascending=False)
    .head(20)
)

display(top_viewed_items)

,itemid,views,addtocarts,transaction_events,unique_viewers,unique_orders
96184,187946,3410,2,0,2911,0
147351,461686,2539,306,133,1379,133
110460,5411,2325,9,0,2078,0
6171,370653,1854,0,0,1577,0
159168,219512,1740,48,12,1358,12
213706,298009,1642,0,0,1342,0
171354,96924,1633,0,0,1359,0
33942,309778,1565,48,15,1053,15
220556,257040,1531,89,27,979,27
51863,384302,1528,65,15,1143,15


In [69]:
top_purchased_items = (
    item_performance
    .sort_values("transaction_events", ascending=False)
    .head(20)
)

display(top_purchased_items)

,itemid,views,addtocarts,transaction_events,unique_viewers,unique_orders
147351,461686,2539,306,133,1379,133
132994,119736,752,44,97,251,96
211651,213834,293,17,92,199,92
29835,312728,947,162,46,602,46
220544,7943,1346,97,46,843,46
119809,445351,939,89,45,618,45
199376,48030,986,95,41,628,41
83131,420960,796,60,38,371,38
150985,248455,575,52,38,291,38
147373,17478,631,72,37,378,37


In [70]:
daily_activity = con.execute("""
    SELECT
        CAST(timestamp AS DATE) AS date,

        COUNT(*) AS total_events,

        COUNT(DISTINCT visitorid) AS unique_users,

        COUNT(*) FILTER (
            WHERE event = 'view'
        ) AS views,

        COUNT(*) FILTER (
            WHERE event = 'addtocart'
        ) AS addtocarts,

        COUNT(*) FILTER (
            WHERE event = 'transaction'
        ) AS transactions

    FROM events
    GROUP BY date
    ORDER BY date
""").df()

display(daily_activity.head(10))

,date,total_events,unique_users,views,addtocarts,transactions
0,2015-05-03,13683,7865,13304,296,83
1,2015-05-04,19414,11019,18681,579,154
2,2015-05-05,23015,13084,22225,565,225
3,2015-05-06,23920,13687,23015,647,258
4,2015-05-07,23164,13086,22369,578,217
5,2015-05-08,19424,11106,18766,532,126
6,2015-05-09,12695,7610,12353,279,63
7,2015-05-10,12143,7413,11771,312,60
8,2015-05-11,16178,9512,15704,390,84
9,2015-05-12,22374,13041,21611,577,186


In [71]:
hourly_activity = con.execute("""
    SELECT
        EXTRACT(HOUR FROM timestamp) AS hour,

        COUNT(*) AS total_events,

        COUNT(DISTINCT visitorid) AS unique_users

    FROM events
    GROUP BY hour
    ORDER BY hour
""").df()

hourly_activity["hour"] = hourly_activity["hour"].astype(int)

display(hourly_activity)

,hour,total_events,unique_users
0,0,144303,89567
1,1,140702,88915
2,2,145879,92636
3,3,150860,96539
4,4,147184,93458
5,5,119572,75804
6,6,76972,49570
7,7,43944,28440
8,8,25309,16918
9,9,17909,12318


In [72]:
order_analysis = con.execute("""
    WITH orders AS (
        SELECT
            transactionid,

            visitorid,

            COUNT(DISTINCT itemid) AS items_in_order

        FROM events

        WHERE event = 'transaction'
          AND transactionid IS NOT NULL

        GROUP BY transactionid, visitorid
    )

    SELECT
        COUNT(DISTINCT transactionid) AS total_orders,

        COUNT(DISTINCT visitorid) AS unique_buyers,

        AVG(items_in_order) AS avg_items_per_order,

        MAX(items_in_order) AS max_items_per_order,

        COUNT(*) FILTER (
            WHERE items_in_order > 1
        ) AS multi_item_orders

    FROM orders
""").df()

display(order_analysis)

,total_orders,unique_buyers,avg_items_per_order,max_items_per_order,multi_item_orders
0,17672,11719,1.263185,31,2643


In [73]:
user_event_times_sql = con.execute("""
    SELECT
        visitorid,

        MIN(timestamp) FILTER (
            WHERE event = 'view'
        ) AS first_view,

        MIN(timestamp) FILTER (
            WHERE event = 'addtocart'
        ) AS first_addtocart,

        MIN(timestamp) FILTER (
            WHERE event = 'transaction'
        ) AS first_transaction

    FROM events

    GROUP BY visitorid
""").df()

/var/folders/gk/l4rj7d3x7wj7wlw6z7x421mr0000gn/T/ipykernel_9414/2068083327.py:20: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()
/var/folders/gk/l4rj7d3x7wj7wlw6z7x421mr0000gn/T/ipykernel_9414/2068083327.py:20: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()
/var/folders/gk/l4rj7d3x7wj7wlw6z7x421mr0000gn/T/ipykernel_9414/2068083327.py:20: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


In [74]:
user_event_times_sql["view_to_cart"] = (
    user_event_times_sql["first_view"].notna()
    & user_event_times_sql["first_addtocart"].notna()
    & (
        user_event_times_sql["first_view"]
        <= user_event_times_sql["first_addtocart"]
    )
)

user_event_times_sql["view_cart_transaction"] = (
    user_event_times_sql["view_to_cart"]
    & user_event_times_sql["first_transaction"].notna()
    & (
        user_event_times_sql["first_addtocart"]
        <= user_event_times_sql["first_transaction"]
    )
)

sql_funnel = pd.DataFrame({
    "stage": [
        "View",
        "Add to cart",
        "Transaction"
    ],

    "users": [
        user_event_times_sql["first_view"].notna().sum(),
        user_event_times_sql["view_to_cart"].sum(),
        user_event_times_sql["view_cart_transaction"].sum()
    ]
})

sql_funnel["conversion_from_view"] = (
    sql_funnel["users"] / sql_funnel.loc[0, "users"]
)

sql_funnel["conversion_from_previous"] = (
    sql_funnel["users"] / sql_funnel["users"].shift(1)
)

sql_funnel.loc[0, "conversion_from_previous"] = 1.0

display(sql_funnel)

,stage,users,conversion_from_view,conversion_from_previous
0,View,1404179,1.000000,1.000000
1,Add to cart,32272,0.022983,0.022983
2,Transaction,9682,0.006895,0.300012


In [75]:
pandas_event_counts = (
    events
    .groupby("event")
    .size()
    .rename("pandas_count")
    .reset_index()
)

sql_event_counts = (
    event_sql[["event", "event_count"]]
    .rename(columns={"event_count": "sql_count"})
)

comparison = pandas_event_counts.merge(
    sql_event_counts,
    on="event",
    how="outer"
)

comparison["difference"] = (
    comparison["pandas_count"]
    - comparison["sql_count"]
)

comparison["match"] = (
    comparison["difference"] == 0
)

display(comparison)

,event,pandas_count,sql_count,difference,match
0,addtocart,69332,69332,0,True
1,transaction,22457,22457,0,True
2,view,2664312,2664312,0,True


In [76]:
assert comparison["match"].all(), (
    "Результаты Pandas и SQL не совпадают!"
)

print("Проверка пройдена: Pandas и SQL дают одинаковые результаты.")

Проверка пройдена: Pandas и SQL дают одинаковые результаты.


In [77]:
sql_funnel_check = user_event_times_sql.copy()

sql_funnel_check["view_to_cart"] = (
    sql_funnel_check["first_view"].notna()
    & sql_funnel_check["first_addtocart"].notna()
    & (
        sql_funnel_check["first_view"]
        <= sql_funnel_check["first_addtocart"]
    )
)

sql_funnel_check["full_funnel"] = (
    sql_funnel_check["view_to_cart"]
    & sql_funnel_check["first_transaction"].notna()
    & (
        sql_funnel_check["first_addtocart"]
        <= sql_funnel_check["first_transaction"]
    )
)

print(
    "Users with view:",
    sql_funnel_check["first_view"].notna().sum()
)

print(
    "Sequential View → Cart:",
    sql_funnel_check["view_to_cart"].sum()
)

print(
    "Sequential full funnel:",
    sql_funnel_check["full_funnel"].sum()
)

Users with view: 1404179
Sequential View → Cart: 32272
Sequential full funnel: 9682


In [78]:
tables_to_export = {
    "11_data_overview.csv": data_overview,
    "11_event_distribution.csv": event_sql,
    "11_user_activity.csv": user_activity,
    "11_user_segments.csv": user_segments,
    "11_item_performance.csv": item_performance,
    "11_top_viewed_items.csv": top_viewed_items,
    "11_top_purchased_items.csv": top_purchased_items,
    "11_daily_activity.csv": daily_activity,
    "11_hourly_activity.csv": hourly_activity,
    "11_order_analysis.csv": order_analysis,
    "11_sequential_funnel.csv": sql_funnel,
    "11_pandas_sql_comparison.csv": comparison
}

for filename, dataframe in tables_to_export.items():
    dataframe.to_csv(
        TABLES_DIR / filename,
        index=False
    )

print(f"Exported {len(tables_to_export)} tables:")
for filename in tables_to_export:
    print(filename)

Exported 12 tables:
11_data_overview.csv
11_event_distribution.csv
11_user_activity.csv
11_user_segments.csv
11_item_performance.csv
11_top_viewed_items.csv
11_top_purchased_items.csv
11_daily_activity.csv
11_hourly_activity.csv
11_order_analysis.csv
11_sequential_funnel.csv
11_pandas_sql_comparison.csv


In [79]:
print("=" * 50)
print("NOTEBOOK 11 — FINAL SUMMARY")
print("=" * 50)

print("Total events:", len(events))
print("Unique users:", events["visitorid"].nunique())
print("Unique items:", events["itemid"].nunique())

print("\nEvent distribution:")
display(event_sql)

print("\nUser segments:")
display(user_segments)

print("\nOrder analysis:")
display(order_analysis)

print("\nSequential funnel:")
display(sql_funnel)

print("\nPandas vs SQL:")
display(comparison)

print("\nAll Pandas and SQL checks passed:",
      comparison["match"].all())

print("\nExported tables:", len(tables_to_export))

NOTEBOOK 11 — FINAL SUMMARY
Total events: 2756101
Unique users: 1407580
Unique items: 235061

Event distribution:


,event,event_count,unique_users,unique_items
0,view,2664312,1404179,234838
1,addtocart,69332,37722,23903
2,transaction,22457,11719,12025



User segments:


,user_segment,users,percentage
0,Viewer,1368715,97.238878
1,Cart user,27146,1.928558
2,Buyer,11719,0.832564



Order analysis:


,total_orders,unique_buyers,avg_items_per_order,max_items_per_order,multi_item_orders
0,17672,11719,1.263185,31,2643



Sequential funnel:


,stage,users,conversion_from_view,conversion_from_previous
0,View,1404179,1.000000,1.000000
1,Add to cart,32272,0.022983,0.022983
2,Transaction,9682,0.006895,0.300012



Pandas vs SQL:


,event,pandas_count,sql_count,difference,match
0,addtocart,69332,69332,0,True
1,transaction,22457,22457,0,True
2,view,2664312,2664312,0,True



All Pandas and SQL checks passed: True

Exported tables: 12
